In [ ]:
#@title << Run this cell first — environment setup {display-mode: "form"}
import sys, os

if "google.colab" in sys.modules:
    !git clone --quiet --single-branch --branch main https://github.com/YanickSchraner/CAS-DeepRL.git
    !cp -r "CAS-DeepRL/Tag 2/envs" .
    sys.path.insert(0, ".")
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

# Unit: Proximal Policy Optimization (PPO) — Dynamic Pricing 🛒💶

## The Real-World Problem

**Dynamic pricing** is one of the highest-ROI applications of AI in business. The core challenge: set the right price at the right time to maximize revenue, given uncertain demand, fluctuating competitor prices, and inventory constraints.

Real-world scale:
- **Amazon** updates prices for millions of products **every 10 minutes**
- **Airlines** re-optimize seat prices thousands of times per day per flight
- **Uber/Lyft** apply real-time surge pricing based on supply/demand signals
- **Hotels** use revenue management systems that increasingly incorporate RL

Traditional rule-based pricing (e.g., *"if inventory < 20%, raise price 15%"*) works but is brittle. RL agents can learn complex multi-variable strategies without explicit rules.

**Your task:** Train a PPO agent to price a product over a 30-day horizon. The agent observes inventory, competitor prices, day of week, and recent demand to set prices that maximize profit.

### 📚 Algorithm: [PPO — Proximal Policy Optimization](https://arxiv.org/abs/1707.06347) via [Stable-Baselines3](https://stable-baselines3.readthedocs.io/)

## Objectives 🏆

By the end of this notebook you will:
- Understand why PPO improves over A2C (the clipping trick)
- Train a PPO agent on a custom business environment
- Plot and **interpret training curves** — the key diagnostic tool for RL engineers
- **Design a better reward function** that captures real business goals
- Compare **fixed pricing strategies** (baselines) against the RL agent

## Install and import

In [ ]:
!pip install stable-baselines3[extra] gymnasium matplotlib --quiet

In [ ]:
import sys, os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import gymnasium as gym
from gymnasium.utils.env_checker import check_env

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.results_plotter import load_results, ts2xy

from envs.pricing_env import PricingEnv

## Part 1 — Understand the Environment

In [ ]:
env = PricingEnv()
check_env(env)

print("=== Pricing Environment ===")
print(f"Observation space : {env.observation_space}")
print(f"  [inventory_frac, competitor_norm, day_sin, day_cos, demand_signal]")
print(f"\nAction space      : {env.action_space}")
print(f"  0=BUDGET (€{PricingEnv.BASE_PRICE * PricingEnv.PRICE_MULT[0]:.0f})  "
      f"1=STANDARD (€{PricingEnv.BASE_PRICE * PricingEnv.PRICE_MULT[1]:.0f})  "
      f"2=PREMIUM (€{PricingEnv.BASE_PRICE * PricingEnv.PRICE_MULT[2]:.0f})  "
      f"3=LUXURY (€{PricingEnv.BASE_PRICE * PricingEnv.PRICE_MULT[3]:.0f})")
print(f"\nEpisode length    : {env.episode_length} days")

obs, _ = env.reset(seed=42)
print(f"\nSample observation:")
print(f"  inventory_frac  : {obs[0]:.2f}  ({obs[0]*100:.0f} units of 100)")
print(f"  competitor_norm : {obs[1]:.2f}")
print(f"  day_sin/cos     : {obs[2]:.2f}, {obs[3]:.2f}")
print(f"  demand_signal   : {obs[4]:.2f}")

## Part 2 — Build Baselines First

**Always establish baselines before training RL agents.** If RL can't beat simple rules, something is wrong (or RL isn't the right tool).

We'll compare three naive strategies:
1. **Always STANDARD price** — no dynamics, just the default price
2. **Always BUDGET price** — always the lowest price
3. **Random** — uniform random pricing
4. **Rule-based** — simple heuristic: high inventory → budget, low inventory → premium

In [ ]:
def evaluate_fixed_policy(action: int, n_episodes: int = 50, seed: int = 0) -> tuple:
    """Evaluate a fixed-action policy."""
    rewards = []
    for ep in range(n_episodes):
        env = PricingEnv(seed=seed + ep)
        obs, _ = env.reset()
        total, done = 0.0, False
        while not done:
            obs, r, term, trunc, _ = env.step(action)
            total += r
            done = term or trunc
        rewards.append(total)
    return float(np.mean(rewards)), float(np.std(rewards))


def evaluate_rule_based(n_episodes: int = 50, seed: int = 0) -> tuple:
    """Heuristic: high inventory → budget, low inventory → premium."""
    rewards = []
    for ep in range(n_episodes):
        env = PricingEnv(seed=seed + ep)
        obs, _ = env.reset()
        total, done = 0.0, False
        while not done:
            inventory_frac = obs[0]
            if inventory_frac > 0.7:
                action = 0   # lots of stock → cut price to sell
            elif inventory_frac > 0.4:
                action = 1   # normal
            elif inventory_frac > 0.2:
                action = 2   # scarce → premium
            else:
                action = 3   # very scarce → luxury
            obs, r, term, trunc, _ = env.step(action)
            total += r
            done = term or trunc
        rewards.append(total)
    return float(np.mean(rewards)), float(np.std(rewards))


baselines = {
    "Random"        : evaluate_fixed_policy(-1, 50),  # we'll handle random below
    "Always BUDGET" : evaluate_fixed_policy(0, 50),
    "Always STANDARD": evaluate_fixed_policy(1, 50),
    "Always PREMIUM" : evaluate_fixed_policy(2, 50),
    "Rule-based"    : evaluate_rule_based(50),
}

# Fix random baseline (needs actual randomness)
rand_rewards = []
for ep in range(50):
    env = PricingEnv(seed=ep)
    obs, _ = env.reset()
    total, done = 0.0, False
    while not done:
        obs, r, term, trunc, _ = env.step(env.action_space.sample())
        total += r; done = term or trunc
    rand_rewards.append(total)
baselines["Random"] = (float(np.mean(rand_rewards)), float(np.std(rand_rewards)))

print("Baseline comparison (mean reward over 30-day episode):")
print("-" * 45)
for name, (mean, std) in baselines.items():
    print(f"  {name:<18}: {mean:8.1f} ± {std:.0f}")

❓ **Question:** Which baseline wins? Why does "Always STANDARD" perform differently from "Always BUDGET"? What does the rule-based heuristic capture that random doesn't?

This is your **target**: your PPO agent should beat the best baseline.

## Part 3 — PPO: What Makes It Better Than A2C?

Both A2C and PPO are policy gradient algorithms. The key difference:

**A2C:** Updates the policy using the gradient directly. Large gradient steps can cause the policy to change too much, collapsing into a bad solution.

**PPO:** Adds a **clipping constraint** that limits how much the policy can change in each update:

```
L_CLIP = min(
    ratio × advantage,
    clip(ratio, 1-ε, 1+ε) × advantage
)
where ratio = π_new(a|s) / π_old(a|s)
```

This makes PPO **more stable** and usually more sample-efficient. With `clip_range=0.2` (default), the policy can change by at most 20% per update step.

**When to use PPO over A2C in practice:**
- Environments with continuous or large action spaces
- When training stability matters (expensive simulations, real hardware)
- PPO is the **industry default** — it's what OpenAI used for Dota2, and what ChatGPT's RLHF used

## Part 4 — Train PPO on the Pricing Environment

In [ ]:
LOG_DIR = "./logs/pricing_ppo/"
os.makedirs(LOG_DIR, exist_ok=True)

train_env = make_vec_env(PricingEnv, n_envs=4)
eval_env = Monitor(PricingEnv(seed=99), LOG_DIR)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=LOG_DIR,
    log_path=LOG_DIR,
    eval_freq=1000,
    n_eval_episodes=20,
    verbose=0,
)

model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    n_steps=256,        # steps per environment per update
    batch_size=64,
    n_epochs=10,        # passes over each batch
    gamma=0.95,         # discount — shorter horizon for pricing (30 days)
    clip_range=0.2,     # the PPO clipping epsilon
    ent_coef=0.01,
    verbose=1,
    seed=42,
)

model.learn(total_timesteps=300_000, callback=eval_callback)
print("Training complete.")

## Part 5 — Evaluate and Plot Training Curves

Training curves are your primary diagnostic tool. Learn to read them.

In [ ]:
# Load evaluation results logged during training
results_path = os.path.join(LOG_DIR, "evaluations.npz")
if os.path.exists(results_path):
    data = np.load(results_path)
    timesteps = data["timesteps"]
    mean_rewards = data["results"].mean(axis=1)
    std_rewards = data["results"].std(axis=1)

    plt.figure(figsize=(12, 5))
    plt.plot(timesteps, mean_rewards, label="PPO (mean)", color="steelblue")
    plt.fill_between(timesteps, mean_rewards - std_rewards, mean_rewards + std_rewards, alpha=0.2)

    # Add baseline lines for comparison
    for name, (m, _) in baselines.items():
        plt.axhline(m, linestyle="--", alpha=0.6, label=name)

    plt.xlabel("Training timesteps")
    plt.ylabel("Episode reward (30-day profit)")
    plt.title("PPO Training Curve vs. Baselines")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    print("No evaluation log found. Run the training cell above first.")

In [ ]:
# Final evaluation
final_mean, final_std = evaluate_policy(model, PricingEnv(seed=200), n_eval_episodes=50)
best_baseline_name = max(baselines, key=lambda k: baselines[k][0])
best_baseline_val = baselines[best_baseline_name][0]

print(f"PPO final performance  : {final_mean:.1f} ± {final_std:.0f}")
print(f"Best baseline          : {best_baseline_name} = {best_baseline_val:.1f}")
print(f"Improvement over best  : {(final_mean - best_baseline_val):.1f} ({(final_mean - best_baseline_val)/abs(best_baseline_val)*100:.1f}%)")

In [ ]:
# Visualise agent pricing decisions over one episode
env_vis = PricingEnv(seed=42)
obs, _ = env_vis.reset()

days, prices, inventory, revenue_per_day, competitor_prices = [], [], [], [], []
done = False

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, term, trunc, info = env_vis.step(int(action))
    done = term or trunc
    days.append(env_vis._day)
    prices.append(info["price"])
    inventory.append(info["inventory"])
    revenue_per_day.append(info["revenue"])
    competitor_prices.append(info["competitor_price"])

dow_labels = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(days, prices, "s-", color="steelblue", label="Agent price")
axes[0].plot(days, competitor_prices, "--", color="orange", alpha=0.7, label="Competitor price")
axes[0].axhline(PricingEnv.BASE_PRICE, color="grey", linestyle=":", label="Base price")
axes[0].set_ylabel("Price (€)")
axes[0].set_title("PPO Pricing Agent — 30-Day Episode")
axes[0].legend()

axes[1].bar(days, inventory, color="green", alpha=0.7)
axes[1].set_ylabel("Inventory (units)")
axes[1].axhline(20, color="red", linestyle="--", label="Restock threshold")
axes[1].legend()

axes[2].bar(days, revenue_per_day, color="steelblue", alpha=0.7)
axes[2].set_ylabel("Daily revenue (€)")
axes[2].set_xlabel("Day")

plt.tight_layout()
plt.show()

print(f"Total revenue: €{sum(revenue_per_day):,.0f}")

❓ **Discuss:**
1. Does the agent price higher on weekends (days 5, 6, 12, 13, ...)? It should — demand is higher.
2. Does the agent respond to inventory levels? Low inventory → higher price?
3. Does the agent undercut the competitor when inventory is high?

These are the three main strategies a good pricing agent should discover.

## Part 6 — Reward Shaping Challenge 🎯

The current reward optimises for **raw profit** (revenue - costs). But real businesses have additional constraints:

**Business constraint:** Marketing says: *"We never want our price to be more than 20% above the competitor. It damages brand perception."*

**Your task:** Add a **constraint penalty** to the reward that discourages pricing above `competitor_price × 1.2`.

This is a very realistic RL engineering problem — adding business constraints to the reward function.

In [ ]:
from envs.pricing_env import PricingEnv

class PricingEnvConstrained(PricingEnv):
    """
    Add a brand-safety constraint: penalize pricing > 20% above competitor.
    """
    BRAND_PENALTY_WEIGHT = 50.0  # how much to penalize a violation

    def step(self, action: int):
        obs, reward, terminated, truncated, info = super().step(action)

        # TODO: Add a penalty when our price > competitor_price * 1.2
        # Hint: info["price"] is our price, info["competitor_price"] is theirs
        constraint_penalty = 0.0  # <-- replace this

        constrained_reward = reward - constraint_penalty
        info["constraint_penalty"] = constraint_penalty

        return obs, constrained_reward, terminated, truncated, info


# Train constrained agent
train_env_c = make_vec_env(PricingEnvConstrained, n_envs=4)
eval_env_c = PricingEnvConstrained(seed=99)

model_c = PPO("MlpPolicy", train_env_c, learning_rate=3e-4, gamma=0.95, verbose=0, seed=42)
model_c.learn(total_timesteps=200_000)

mean_c, std_c = evaluate_policy(model_c, eval_env_c, n_eval_episodes=50)
print(f"Constrained agent reward : {mean_c:.1f} ± {std_c:.0f}")
print(f"Unconstrained agent      : {final_mean:.1f} ± {final_std:.0f}")
print("(Constrained agent should earn less but stay within 20% of competitor)")

In [ ]:
# Check how often the constrained agent violates the rule
env_check = PricingEnvConstrained(seed=42)
obs, _ = env_check.reset()
violations = 0
total_steps = 0
done = False

while not done:
    action, _ = model_c.predict(obs, deterministic=True)
    obs, _, term, trunc, info = env_check.step(int(action))
    done = term or trunc
    if info["price"] > info["competitor_price"] * 1.2:
        violations += 1
    total_steps += 1

print(f"Constraint violations: {violations}/{total_steps} steps ({violations/total_steps*100:.1f}%)")

❓ **Reflect:**
1. Did your penalty weight eliminate violations or just reduce them?
2. Is the constrained agent's profit significantly lower? Is the trade-off worth it?
3. How would you determine the right `BRAND_PENALTY_WEIGHT` in a real business setting?

## Part 7 — Training Curve Interpretation: What Went Wrong? 🔍

Below are three training curves. **Without running any code**, diagnose what each one shows:

In [ ]:
# Simulate three training curves with different failure modes
x = np.linspace(0, 300_000, 300)

# Good training: steady improvement then plateau
curve_good = -2000 * np.exp(-x / 80_000) + 500 + np.random.normal(0, 30, 300)

# Learning rate too high: improves then collapses
curve_collapse = np.where(
    x < 120_000,
    -1500 * np.exp(-x / 60_000) + 200,
    200 - 800 * (x - 120_000) / 180_000
) + np.random.normal(0, 80, 300)

# Epsilon too low (no exploration): stuck at bad local optimum early
curve_stuck = -1800 * np.exp(-x / 20_000) - 900 + np.random.normal(0, 20, 300)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, curve_good, label="Agent A", color="green")
ax.plot(x, curve_collapse, label="Agent B", color="red")
ax.plot(x, curve_stuck, label="Agent C", color="orange")
ax.axhline(baselines["Always STANDARD"][0], linestyle="--", color="grey", label="Standard baseline")
ax.set_xlabel("Training timesteps")
ax.set_ylabel("Episode reward")
ax.set_title("Three Training Runs — Diagnose the Failures")
ax.legend()
plt.tight_layout()
plt.show()

❓ **Diagnostic exercise** (no code needed):

For each agent, answer:
1. **Agent A**: Describe what happened. Is this a successful run?
2. **Agent B**: The reward improves, then drops sharply. What likely caused this? What hyperparameter would you change?
3. **Agent C**: The reward plateaus very quickly at a low value and never improves. What's the likely cause?

Refer to what you learned about `learning_rate`, `ent_coef`, and `clip_range` in this notebook.

## Summary

| What you did | RL Engineer skill |
|---|---|
| Built multiple baselines before training | Never skip baselines — they tell you if RL is needed |
| Understood PPO vs. A2C clipping | Algorithm selection based on stability needs |
| Plotted and read training curves | Primary diagnostic tool in production RL |
| Added a business constraint to the reward | Real-world RL requires translating business rules into reward components |
| Diagnosed failure curves | RL debugging: identify failure mode → fix hyperparameter |

### RL Engineer Toolkit — what you've now used
| Tool | Purpose |
|---|---|
| SB3 `PPO`, `A2C` | Production-quality algorithm implementations |
| `make_vec_env` | Parallelise data collection |
| `EvalCallback` | Track progress, auto-save best model |
| `evaluate_policy` | Unbiased final evaluation |
| `Monitor` | Log episode rewards for curve plotting |
| Custom `gymnasium.Env` subclass | Build domain-specific environments |

**Next:** RLHF — how PPO was used to train ChatGPT and Claude. 🤖